# Content Refresh Prioritization — Capstone Research Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikasbit/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This capstone notebook summarizes the research question, data, methodology, results, limitations, and action playbook for Content Refresh Prioritization.


## 1. Question

**Research Question:** Which webpages should be prioritized for content refresh based on historical search performance and content signals?

**Decision Supported:** Allocating editorial refresh resources to high-impact decaying pages vs publishing net-new content.


In [ ]:
import json
import pandas as pd

with open("outputs/summary.json") as f:
    summary = json.load(f)

print("Rows scored:", summary["rows_scored"])
print("Best model:", summary["best_model"])
print("Target positive rate:", round(summary["target_positive_rate"], 3))


## 2. Data

- Dataset: Bundled anonymized FlyRank dataset (`content_refresh_anonymized.csv`).
- Rows: 30,000 scored rows (27,675 train / 2,325 test split across 32 clients).
- Target: `is_declining_label` (54.21% base rate).
- Exclusions: Pseudonymized IDs (`content_id`, `client_id`), no private URLs or query text.


In [ ]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 3. Methodology

- Feature Construction: 52 features (18 numeric, 8 categorical One-Hot encoded).
- Split Strategy: Client-holdout split (`client_holdout`).
- Models Evaluated: Baseline Rules, Logistic Regression, Decision Tree, Random Forest.
- Leakage Audit: Zero forward-window target leaks.


In [ ]:
with open("outputs/model_results.json") as f:
    res = json.load(f)

print("Split strategy:", res["split_strategy"])
print("Feature count:", res["feature_count"])


## 4. Results (vs baseline)

### Model Comparison Table

| Method | Validation | ROC AUC | Avg Precision | Precision@50 | Recall | F1 Score |
|---|---|---:|---:|---:|---:|---:|
| Week-4 Baseline | Client-holdout split | 0.627 | 0.468 | 0.240 | 0.189 | 0.274 |
| Decision Tree | Client-holdout split | 0.742 | 0.575 | 0.620 | 0.716 | 0.634 |
| Logistic Regression | Client-holdout split | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| **Random Forest (Best)** | Client-holdout split | **0.747** | **0.610** | **0.680** | **0.741** | **0.638** |
| Week-6 Honest Validation | Grouped/time-aware | 0.747 | 0.610 | 0.680 | 0.741 | 0.638 |


In [ ]:
models = res["models"]
b = res["baseline"]
print(f"Baseline Precision@50: {b['baseline_precision_at_50']:.3f}")
print(f"Random Forest Precision@50: {models['random_forest']['precision_at_50']:.3f}")
print(f"Random Forest ROC AUC: {models['random_forest']['roc_auc']:.3f}")


## 5. Limitations

- Observational data only; no causal guarantees.
- Seasonal traffic variations not fully modeled in 90-day window.
- **Mandatory Statement:** *This analysis provides directional decision-support and does not establish that refreshing a page will cause improved traffic, rankings, or conversions.*


## 6. Ranked recommendations

- `CTR_OPPORTUNITY` → `REFRESH_CONTENT`
- `HIGH_DEMAND` → `REVIEW_HIGH_DEMAND`
- `RANKING_OPPORTUNITY` → `REVIEW_RANKING`
- `LOWER_PRIORITY` → `MONITOR`


In [ ]:
queue = pd.read_csv("outputs/refresh_queue.csv")
print("Suggested Actions Summary:")
print(queue["suggested_action"].value_counts())


## 7. Artifacts the paper embeds

- `outputs/charts/action_mix.svg`
- `outputs/charts/confidence_mix.svg`
- `outputs/charts/top_reason_codes.svg`
- `outputs/charts/top_feature_importance.svg`
- `outputs/charts/trend_distribution.svg`


## ML-12 Summary & Presentation

### 5-Minute Demo Outline
1. **Problem:** Content decay hurts organic traffic; manually auditing 30,000 pages is impossible.
2. **Data & Leakage Audit:** Built 52 leakage-free pre-decision features from GSC/GA4 signals.
3. **Baseline vs Model:** Baseline rules achieved 0.240 Precision@50; Random Forest improved this to 0.680.
4. **Action Playbook:** Automated queue flags CTR and ranking opportunities for human review.
5. **Honest Framing:** Decision-support tool for editorial review, not automated publishing.

### Social-Post Cut
🚀 Excited to share my FlyRank ML Internship capstone on Content Refresh Prioritization!
Using 30k anonymized search performance rows across 32 clients, our Random Forest model achieved a 0.747 ROC AUC and 0.680 Precision@50 (vs 0.240 baseline rules) to flag decaying content for editorial refresh.
Check out the live paper: https://vikasbit.github.io/flyrank-ml-internship/
Built on data from FlyRank (https://flyrank.ai)

### 3-Sentence Employer Summary
- Built an end-to-end Machine Learning pipeline ranking 30,000 pages for content refresh using client-holdout validation to ensure generalization across unseen domains.
- Achieved a 0.747 ROC AUC and a 2.8× precision improvement over heuristic rules (0.680 vs 0.240 Precision@50) without target leakage.
- Deployed a public, transparent research paper detailing methodological rigor, model governance, and honest decision-support limitations.
